# 03 — Historical AmazonHelp Support Corpus & Retrieval Baseline

**Phase 5 of 9** for the Hiver SDE-intern take-home. Phase 1 selected **AmazonHelp**; Phase 2 discovered the intent taxonomy; Phases 3–4 shipped a gold set and weak deterministic classifiers. This notebook asks the RAG question: **does the ~203k historical interaction corpus help an agent recall how a brand-side agent actually handled a comparable interaction?** It evaluates *retrieval only* — no agent, no generated replies, no escalation policy. Phases 6–7 later consume the retrieved resolutions for drafting and escalation.

**Method (fully reproducible, deterministic seed 42, no API key):**
1. Assemble the conversation-aware interaction corpus (`customer message → context → brand    response`), excluding the golden + holdout conversations with a hard assertion.
2. Quantify corpus facts that shape retrieval (multilinguality, canned/template replies,    URL-heavy messages, context dependency).
3. Build TF-IDF, BM25, dense (all-MiniLM-L6-v2) and hybrid retrievers over message-only vs    message+context inputs; evaluate against a **hand-labelled 30-query / 150-pair** pool with    a temporal filter (retrieved interactions must predate the tweet).
4. Measure the **data-scaling curve** on nested subsets (10k / 50k / 100k / full).
5. Categorize >=30 retrieval failures and publish them with raw evidence.

The notebook recomputes every number live from committed artifacts; it does not re-embed or re-train anything (those live in the scripts).

## 1. Corpus assembly & leak protection

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
import numpy as np
sys.path.insert(0, '..')
from src import config
from src.retrieval.corpus import load_corpus, load_holdout_ids

corpus_path = config.DATA_DIR / 'retrieval' / 'corpus.slim.parquet'
df = pd.read_parquet(corpus_path)
print('corpus rows:', len(df))
print('usable responses: {:.1f}%'.format(100 * df['has_usable_response'].mean()))
print('distinct conversations:', df['conversation_id'].nunique())
print('canned templates (>=20 copies): {:.2f}%'.format(100 * df['is_canned_template'].mean()))
print(df['language'].value_counts().head(8).to_string())


corpus rows: 201741
usable responses: 89.1%
distinct conversations: 82256
canned templates (>=20 copies): 1.87%
language
en    157157
es     12889
ja      9921
fr      7772
pt      6152
de      5329
it      1879
nl       309


In [2]:
holdout_conv = {int(x) for x in load_holdout_ids()}
indexed_conv = set(df['conversation_id'].astype(int))
overlap = indexed_conv & holdout_conv
print('golden/holdout conversations excluded from index:', len(overlap), '| (must be 0)')
assert len(overlap) == 0
print('leak check PASSED')


golden/holdout conversations excluded from index: 0 | (must be 0)
leak check PASSED


## 2. The retrieval benchmark

A pool of 30 queries (24 English + 6 non-English, all treated as context-dependent) was drawn from golden/holdout-adjacent material with seed 42, and 150 candidate pairs were hand-labelled *0 = irrelevant, 1 = related, 2 = useful*. Recall is pool-based (unjudged top-K hits count as misses — standard pooling); hybrid alphas are reported, not tuned on golden labels.

In [3]:
R = dict()
rep = json.loads(Path('../reports/retrieval_results.json').read_text())
import csv
rows = list(csv.DictReader(open('../reports/retrieval_results.csv')))
print('{:<12} {:<4} {:>5} {:>5} {:>5} {:>5} {:>5} {:>5} {:>7}'.format(
    'model', 'in', 'R@1', 'R@3', 'R@5', 'R@10', 'MRR', 'useful5', 'lat_ms'))
for r in rows:
    print('{:<12} {:<4} {:>5.3f} {:>5.3f} {:>5.3f} {:>5.3f} {:>5.3f} {:>5.3f} {:>7.1f}'.format(
        r['model'], r['variant'], float(r['R@1']), float(r['R@3']), float(r['R@5']),
        float(r['R@10']), float(r['MRR']), float(r['useful_R@5']), float(r['latency_ms_avg'])))
print()
print('pooling note:', rep['pooling_notes'])


model        in     R@1   R@3   R@5  R@10   MRR useful5  lat_ms
bm25         ctx  0.076 0.109 0.109 0.164 0.239 0.081   300.8
bm25         msg  0.134 0.190 0.203 0.276 0.447 0.189   100.2
dense        ctx  0.184 0.302 0.358 0.366 0.683 0.403  2431.7
dense        msg  0.252 0.448 0.576 0.593 0.850 0.583   268.4
hybrid       ctx  0.087 0.117 0.152 0.192 0.270 0.133   782.7
hybrid_a03   ctx  0.076 0.111 0.134 0.156 0.232 0.125   720.0
hybrid_a07   ctx  0.054 0.145 0.202 0.257 0.311 0.194   535.7
hybrid       msg  0.164 0.277 0.325 0.411 0.533 0.358   473.4
hybrid_a03   msg  0.137 0.254 0.299 0.327 0.461 0.297   294.0
hybrid_a07   msg  0.154 0.353 0.426 0.552 0.632 0.444   368.4
tfidf        ctx  0.132 0.183 0.217 0.241 0.468 0.206   208.8
tfidf        msg  0.063 0.194 0.241 0.280 0.388 0.214   123.8

pooling note: R@K is pool-based: candidates in the judged pool are the only positives/unjudged top-K hits count as misses. Hybrid alphas are reported for every value; alpha is NOT tuned on go

## 3. Data-scaling curve (message-only)

In [4]:
sc = json.loads(Path('../reports/retrieval_scaling.json').read_text())
print('seed', sc['seed'], '| nested subsets | size rows:')
prev = {}
for r in sc['rows']:
    prev[r['model']] = r['R@5']
    print('  {:5s} n={:>7d} R@5={:.3f} MRR={:.3f} lat={:.0f}ms idx={:.0f}MB'.format(
        r['model'], r['corpus_size'], r['R@5'], r['MRR'], r['lat_ms_avg'], r['index_bytes']/1e6))
d = [r for r in sc['rows'] if r['model'] == 'dense']
g = (d[3]['R@5'] - d[1]['R@5']) / max(1e-9, (d[3]['corpus_size'] - d[1]['corpus_size']))
print('\ndense R@5 last-window slope (50k->full): {:.3f} per 1000 docs (still climbing -> no plateau)'.format(g*1000))


seed 42 | nested subsets | size rows:
  bm25  n=  10000 R@5=0.011 MRR=0.017 lat=7ms idx=1MB
  dense n=  10000 R@5=0.028 MRR=0.100 lat=901ms idx=15MB
  bm25  n=  50000 R@5=0.088 MRR=0.196 lat=89ms idx=7MB
  dense n=  50000 R@5=0.192 MRR=0.525 lat=215ms idx=77MB
  bm25  n= 100000 R@5=0.104 MRR=0.259 lat=52ms idx=14MB
  dense n= 100000 R@5=0.315 MRR=0.711 lat=89ms idx=154MB
  bm25  n= 201741 R@5=0.203 MRR=0.433 lat=305ms idx=28MB
  dense n= 201741 R@5=0.576 MRR=0.850 lat=544ms idx=310MB

dense R@5 last-window slope (50k->full): 0.003 per 1000 docs (still climbing -> no plateau)


## 4. Error analysis (>=30 categorized failures)

In [5]:
ea = json.loads(Path('../reports/retrieval_error_analysis.json').read_text())
print('failures:', ea['n_failures'], '| by mode:')
for k, v in sorted(ea['by_mode'].items(), key=lambda x: -x[1]):
    print('  {:~<34}'.format(k), v)
print('\nheadline (dense/msg) failures:')
for f in [x for x in ea['failures'] if x['model'] == 'dense' and x['input_variant'] == 'msg'][:8]:
    print('  {} {} {} -> missed {}'.format(f['query_id'], f['mode'], f['miss_kind'],
                                          f['missed_useful_case_ids'][:3]))


failures: 131 | by mode:
  lexical_mismatch~~~~~~~~~~~~~~~~~~ 99
  semantic_near_miss~~~~~~~~~~~~~~~~ 14
  multilingual_crossover~~~~~~~~~~~~ 12
  benchmark_absence~~~~~~~~~~~~~~~~~ 5
  insufficient_context_handling~~~~~ 1

headline (dense/msg) failures:
  challenge-1379340 insufficient_context_handling partial_useful_miss -> missed ['2317689', '139027']
  challenge-1568158 semantic_near_miss partial_useful_miss -> missed ['538444']
  challenge-2694949 semantic_near_miss partial_useful_miss -> missed ['476110']
  challenge-842315 semantic_near_miss partial_useful_miss -> missed ['162020']
  representative-1136222 semantic_near_miss partial_useful_miss -> missed ['2496352']
  representative-1179639 semantic_near_miss full_useful_miss -> missed ['577708']
  representative-1210214 semantic_near_miss partial_useful_miss -> missed ['2873344', '98571']
  representative-2217342 semantic_near_miss partial_useful_miss -> missed ['1514354', '2598955']


## 5. Reproduce (deterministic, no API key)

In [6]:
print('''
python scripts/build_retrieval_index.py --models tfidf,bm25,dense,hybrid --variant both
python scripts/prepare_relevance_benchmark.py
python scripts/evaluate_retrieval.py
python scripts/build_weak_intents.py
python scripts/evaluate_retrieval_scaling.py
python scripts/build_error_analysis.py
''')



python scripts/build_retrieval_index.py --models tfidf,bm25,dense,hybrid --variant both
python scripts/prepare_relevance_benchmark.py
python scripts/evaluate_retrieval.py
python scripts/build_weak_intents.py
python scripts/evaluate_retrieval_scaling.py
python scripts/build_error_analysis.py

